Felhasználó megadja a játékos nevét -> a program kimutatja a rá vonatkozó statisztikákat.

3 MVP:

1.   Legsikeresebb pályák / Autó osztályok.
2.   Összes ELO +/- változás.
3.   Pódium Ráta.




**HASZNOS LINKEK.**

*Félreértések elkerülése végett: a {} közötti értékek user / kód által megadott értékek.*


*   Json link az user ranked ID-ivel:

    https://game.raceroom.com/users/{username}/career?CurrentPage=-{Page}&PageSize=100&json

    NOTE TO SELF: RaceHash eleresi utja jsonon at:
    
    ["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"]

*   Json link egy ranked verseny eredményeihez:

    https://game.raceroom.com/multiplayer/results/{Race_ID}

*   Json link Car/Track infokhoz:
    https://raw.githubusercontent.com/sector3studios/r3e-spectator-overlay/master/r3e-data.json


*   User profilja:

    https://game.raceroom.com/users/{username}/

*   User karrier oldala:

    https://game.raceroom.com/users/{username}/career

*   User specifikus ranked infók json-je:

    NOTE TO SELF: ez csak User ID-val múködik, username-el nem.

    https://game.raceroom.com/multiplayer-rating/user/{player_id}.json
*   User specifikus non-ranked adatok+extrák:

    https://game.raceroom.com/utils/user-info/{username}
*   UNIX CONVERTER:

    https://www.epochconverter.com/
    
    Sat, 25 Sep 2025 13:34:33 UTC -> 1758807273 (25 Q3 RELEASE DATE, important bcoz there were a lot of changes such as the removal of Q sessions from Sprint Races)

**Osztályok:**

In [2]:
#region Import
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import List
#endregion

#region Player
@dataclass
class Player:
    user_id: str
    username: str
    full_name: str
    country: str
    elo: float
    rep: float
    race_count: int
    global_position: int

#endregion

#region Race Meta
@dataclass
class Race_Meta:
    race_id: str #done
    #region actual metadata
    date: datetime #done
    start_position: int
    start_position_in_class: int
    finish_position: int
    finish_position_in_class: int
    player_count: int #done
    incident_points: int
    reputation_after: float
    reputation_change: float
    rating_after: float
    rating_change: float
    track: str #done
    track_layout: str #done
    #endregion

def Race_Meta_Is_After_Q3_Release(Race_Meta) -> bool:
  Q3_release = 1758807273
  Q3_release = datetime.fromtimestamp(Q3_release, tz = gmt_plus_2).strftime('%Y.%m.%d %H:%M')
  if Race_Meta.date > Q3_release:
    return True
  else:
    return False

#endregion

#region Qualifying
@dataclass
class Qualifying_Lap:
    username: str
    race_id: str
    position: int
    position_In_Class: int
    time: datetime
    sectortimes: str

#endregion

#region Race
@dataclass
class Race_Lap:
    username: str
    race_id: str
    position: int
    position_In_Class: int
    time: datetime
    sectortimes: str

#endregion Race

#region Result
@dataclass
class Result:
    player: Player
    race_meta: Race_Meta
#endregion Race

In [4]:
#region Import
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from random import randint
from typing import List
import urllib.request
import requests
import statistics
import json
import math
import random

#endregion

#region ország-időzóna leképezés
country_to_timezone = {
    "Hungary": "Europe/Budapest",
    "Germany": "Europe/Berlin",
    "France": "Europe/Paris",
    "United Kingdom": "Europe/London",
    "United States": "America/New_York",
    # vagy más régió
    # Bővíthető további országokkal
}

def get_timezone_from_country(country_name: str) -> ZoneInfo:
    tz_name = country_to_timezone.get(country_name)
    if tz_name:
        return ZoneInfo(tz_name)
    else:
        # Alapértelmezett időzóna, ha nincs találat
        return ZoneInfo("UTC")

#endregion

#region Teszt Input-ok
Orban_Krisztian = "orban_k"
en = "Bab_0"
Czari_Patrik = "WEaSeL_4"
Vilmos_Bajusz = "danubajusz"
Frank_Giesler = "bankmann_0"
Daniel_Bolyki = "bossofcats"
players = [Orban_Krisztian, en, Czari_Patrik, Vilmos_Bajusz, Frank_Giesler, Daniel_Bolyki]
#endregion

#region DONE

#region GPD
def get_player_data(username: str) -> Player:
  #region variables
  user_id: str
  full_name: str
  country: str
  elo: float
  rep: float
  race_count: int
  global_position: int
  #endregion

  basic_user_info = "https://game.raceroom.com/utils/user-info/{0}".format(username)

  with urllib.request.urlopen(basic_user_info) as response:
    data = json.load(response)

    user_id = data["id"]
    full_name = data["name"]
    country = data["country"]["name"]

  ranked_user_info = "https://game.raceroom.com/multiplayer-rating/user/{0}.json".format(user_id)

  with urllib.request.urlopen(ranked_user_info) as response:
    data = json.load(response)
    elo = data["Rating"]
    rep = data["Reputation"]
    race_count = data["RacesCompleted"]
    if "Position" in data:
      global_position = data["Position"]
    else:
      global_position = None

  return Player(user_id, username, full_name, country, elo, rep, race_count, global_position)

#endregion

#region RMD
def get_race_meta_data(username, page) -> List[Race_Meta]:
  if page > 100:
      page = round(-round(page/100))
  elif page < 100:
    page = -1
  #region vars
  race_id: str

  date: datetime
  #region testing atm
  start_position: int
  start_position_in_class: int
  finish_position: int
  finish_position_in_class: int
  #endregion

  player_count: int
  #region testing atm
  incident_points: int
  reputation_after: float
  reputation_change: float
  rating_after: float
  rating_change: float
  #endregion

  track: str
  track_layout: str
  #vars
  mylist: List[Race_Meta] = []
  #lst
  #endregion

  for page in range(page,0):
    url = "https://game.raceroom.com/users/{0}/career?CurrentPage={1}&PageSize=100&json".format(username, page)
    with urllib.request.urlopen(url) as response:
      data = json.load(response)
      #CIKLUSSAL VÉGIG OLVASÁS !!!
      for i in range(len(data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"])):
        # !!!
        #region reading JSONs
        race_id = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["RaceHash"]

        date = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["RaceFinishTime"]

        #region under testing atm
        start_position = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["StartPosition"]

        start_position_in_class = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["StartPositionInClass"]

        finish_position = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["FinishPosition"]

        finish_position_in_class = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["FinishPositionInClass"]
        #endregion

        player_count = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["PlayersCount"]

        #region under testing atm
        incident_points = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["IncidentPoints"]

        reputation_after = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["ReputationAfter"]

        reputation_change = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["ReputationChange"]

        rating_after = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["RatingAfter"]

        rating_change = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["RatingChange"]
        #endregion

        track = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["TrackLayoutId"]["Id"]

        track_layout = data["context"]["c"]["raceList"]["GetUserMpRatingProgressResult"]["Entries"][i]["TrackLayoutId"]["Name"]

        if race_id not in mylist:
          mylist.append(Race_Meta(race_id, date, start_position, start_position_in_class, finish_position, finish_position_in_class, player_count, incident_points, reputation_after, reputation_change, rating_after, rating_change, track, track_layout))
  return mylist
#endregion

#region GQL
def get_quali_laps(race_id: str, username: str):
  #region variables
  position: int
  position_in_class: int
  time: datetime
  #endregion
  lap_lst = []
  url = f"https://game.raceroom.com/multiplayer/results/{race_id}"
  with urllib.request.urlopen(url) as response:
    data = json.load(response)
    for i in range(len(data["GetMpRaceResultResult"]["QualiResult"])):
      if data["GetMpRaceResultResult"]["QualiResult"][i]["Username"] == username:
        for j in range(len(data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"])):
          #region load data into variables
          position = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["Position"]
          position_in_class = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["PositionInClass"]
          time = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["Time"]
          Sectortimes = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["SectorTimes"]
          #endregion
          lap_lst.append(Qualifying_Lap(username, race_id, position, position_in_class, time, Sectortimes))
    return lap_lst
#endregion

#region GRL

def get_race_laps(race_id: str, username: str):
  #region variables
  position: int
  position_in_class: int
  time: datetime
  #endregion
  lap_lst = []
  url = f"https://game.raceroom.com/multiplayer/results/{race_id}"
  with urllib.request.urlopen(url) as response:
    data = json.load(response)
    for i in range(len(data["GetMpRaceResultResult"]["RaceResult"])):
      if data["GetMpRaceResultResult"]["RaceResult"][i]["Username"] == username:
        for j in range(len(data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"])):
          #region load data into variables
          position = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["Position"]
          position_in_class = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["PositionInClass"]
          time = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["Time"]
          Sectortimes = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["SectorTimes"]
          #endregion
          lap_lst.append(Race_Lap(username, race_id, position, position_in_class, time, Sectortimes))
    return lap_lst

#endregion

#endregion

#region Test
curious = True
if curious == True:
  #region data handling
  myplayers: List[Player] = []
  p = 0
  for p in range(len(players)):
    myplayers.append(get_player_data(players[p]))

  Krisztian = myplayers[0]
  me = myplayers[1]
  Patrik = myplayers[2]
  Vilmos = myplayers[3]
  Frank = myplayers[4] #no. 1 player afaik
  Daniel = myplayers[5]
  player = me

  my_race_meta = sorted(get_race_meta_data(player.username, player.race_count), key=lambda Race_Meta: -Race_Meta.date)

  rating_gain_lst = []
  sum_of_rating_gain = 0
  sum_of_rating_loss = 0
  for row in my_race_meta:

    if row.rating_change > 0:
      rating_gain_lst.append(row.rating_change)
      sum_of_rating_gain += row.rating_change
    else:
      sum_of_rating_loss += row.rating_change

  avg_rating_gain = round(sum_of_rating_gain/len(my_race_meta), 3)
  stdev_of_rating_gain = round(statistics.stdev(rating_gain_lst), 3)
  consistency = round(100-((stdev_of_rating_gain/max(rating_gain_lst))*100), 3)
  #endregion

  #region print basic data
  print(f"{player.full_name} ({player.username}, #{player.user_id}) - {player.country}\nRace count: {player.race_count}\nRating: {player.elo}\nReputation {player.rep}\n#{player.global_position} on the Global LB.\nAverage rating gain: {avg_rating_gain}\nStandard Deviation of rating gain {stdev_of_rating_gain}\nConsistency of rating gain: {consistency} %")

  print(f"\nPast race results for {player.full_name}\n- WORK IN PROGRESS - \n")
  #endregion

  for row in my_race_meta:

    local_tz = get_timezone_from_country(me.country)
    row.date = datetime.fromtimestamp(row.date, tz = local_tz).strftime('%Y.%m.%d %H:%M')

    print(f"---{row.date}---\nSPOS: P{row.start_position}(P{row.start_position_in_class} in class)\nFPOS: P{row.finish_position} (P{row.finish_position_in_class} in class)\nRATING: {row.rating_after} ({row.rating_change})\nREPUTATION: {row.reputation_after} ({row.reputation_change})\nINCIDENT POINTS: {row.incident_points} \n --- _ ---\n")


  #endregion
#endregion
#endregion

#WORK IN PROGRESS

Benedek Szabó (Bab_0, #6524740) - Hungary
Race count: 265
Rating: 1608.004
Reputation 74.111
#2968 on the Global LB.
Average rating gain: 5.664
Standard Deviation of rating gain 8.057
Consistency of rating gain: 76.759 %

Past race results for Benedek Szabó
- WORK IN PROGRESS - 

---2025.10.06 21:40---
SPOS: P8(P8 in class)
FPOS: P14 (P14 in class)
RATING: 1608.004 (-1.344)
REPUTATION: 74.111 (0.755)
INCIDENT POINTS: 21 
 --- _ ---

---2025.10.06 21:09---
SPOS: P1(P1 in class)
FPOS: P21 (P21 in class)
RATING: 1609.348 (-46.2)
REPUTATION: 73.356 (-8.04)
INCIDENT POINTS: 3 
 --- _ ---

---2025.10.06 20:38---
SPOS: P2(P2 in class)
FPOS: P3 (P3 in class)
RATING: 1655.548 (20.343)
REPUTATION: 81.396 (0.72)
INCIDENT POINTS: 12 
 --- _ ---

---2025.10.06 19:10---
SPOS: P7(P7 in class)
FPOS: P13 (P13 in class)
RATING: 1635.205 (-16.463)
REPUTATION: 80.676 (-0.378)
INCIDENT POINTS: 25 
 --- _ ---

---2025.10.06 13:07---
SPOS: P7(P7 in class)
FPOS: P11 (P11 in class)
RATING: 1651.668 (-17.283)
R

In [ ]:
#region IMPORT
from datetime import time, datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from random import randint
from typing import List
import urllib.request
import requests
import json
import math
import random
#endregion

#region Get Quali Laps
def get_quali_laps(race_id: str, username: str):
  #region variables
  position: int
  position_in_class: int
  time: datetime
  #endregion
  lap_lst = []
  url = f"https://game.raceroom.com/multiplayer/results/{race_id}"
  with urllib.request.urlopen(url) as response:
    data = json.load(response)
    for i in range(len(data["GetMpRaceResultResult"]["QualiResult"])):
      if data["GetMpRaceResultResult"]["QualiResult"][i]["Username"] == username:
        for j in range(len(data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"])):
          #region load data into variables
          position = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["Position"]
          position_in_class = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["PositionInClass"]
          time = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["Time"]
          Sectortimes = data["GetMpRaceResultResult"]["QualiResult"][i]["Laps"][j]["SectorTimes"]
          #endregion
          lap_lst.append(Qualifying_Lap(username, race_id, position, position_in_class, time, Sectortimes))
    return lap_lst
#endregion

#region Get Race Laps

def get_race_laps(race_id: str, username: str):
  #region variables
  position: int
  position_in_class: int
  time: datetime
  #endregion
  lap_lst = []
  url = f"https://game.raceroom.com/multiplayer/results/{race_id}"
  with urllib.request.urlopen(url) as response:
    data = json.load(response)
    for i in range(len(data["GetMpRaceResultResult"]["RaceResult"])):
      if data["GetMpRaceResultResult"]["RaceResult"][i]["Username"] == username:
        for j in range(len(data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"])):
          #region load data into variables
          position = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["Position"]
          position_in_class = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["PositionInClass"]
          time = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["Time"]
          Sectortimes = data["GetMpRaceResultResult"]["RaceResult"][i]["Laps"][j]["SectorTimes"]
          #endregion
          lap_lst.append(Race_Lap(username, race_id, position, position_in_class, time, Sectortimes))
    return lap_lst

#endregion

#region Test

#region Race Test
print("Race results:")
racelap = 0
myraceresults = get_race_laps("60rnzn", "Bab_0")
for i in myraceresults:

  if i.time > 0:
    mytime = str(timedelta(milliseconds=i.time))[3:-2]
    if i.time > 600000:
      mytime = str(timedelta(milliseconds=i.time))[2:-2]
  else:
    mytime = "Invalid"

  racelap+=1
  print(f"L{racelap}, P{i.position}: {mytime}\n")

#endregion


#region Quali Test
print("Quali results:")
lapnum = 0
myqualilaps = get_quali_laps("60rnzn", "Bab_0")
for i in myqualilaps:
  lapnum+=1
  s1 = i.sectortimes[0]
  s2 = i.sectortimes[1]-s1
  s3 = i.sectortimes[2]-s2-s1
  #region Qlaptime test
  if i.time > 0:
    mytime = str(timedelta(milliseconds=i.time))[3:-3]
    if i.time > 600000:
      mytime = str(timedelta(milliseconds=i.time))[2:-4]
  else:
    mytime = "Invalid"
  #endregion

  #region Qsectortime test
  if s1 > 0 and s2 > 0 and s3 > 0:
    s1 = str(timedelta(milliseconds=s1))[5:-4]
    s2 = str(timedelta(milliseconds=s2))[5:-4]
    s3 = str(timedelta(milliseconds=s3))[5:-4]
    print(f"L{lapnum}, P{i.position}: {mytime}\nS1({s1})  S2({s2})  S3({s3})\n")
  else:
    print(f"L{lapnum}, P{i.position}: {mytime}\n")

#endregion
#endregion

#endregion
  # test race id for this function is: w12w34, test user is: me, Qualified 2nd, tho i was 1st for a few laps during the session. This is done so we can see if we log diff. laps w/ diff. pos.

  #testing with a race where I had no full laps during Q (or there were no Q session at all, such as VoyrQM)threw an error, gotta fix this

Race results:
L1, P9: 1:09.6670

L2, P9: 1:00.7110

L3, P9: 1:00.4530

L4, P10: 1:01.0850

L5, P11: 0:59.9690

L6, P11: 0:59.4590

L7, P11: 1:00.7320

L8, P12: 1:00.6820

L9, P12: 1:01.2090

L10, P12: 1:00.2350

L11, P12: 0:59.6300

L12, P11: 1:00.3300

L13, P11: 1:00.1080

L14, P12: 1:

L15, P12: 1:01.3320

L16, P11: 0:59.5770

Quali results:
L1, P8: Invalid

L2, P3: 0:59.408
S1(24.22)  S2(19.93)  S3(15.24)

L3, P3: 1:00.388
S1(24.82)  S2(20.26)  S3(15.29)

L4, P4: 1:00.077
S1(24.54)  S2(20.17)  S3(15.35)

L5, P8: Invalid

L6, P6: 1:00.458
S1(25.13)  S2(20.02)  S3(15.29)

L7, P8: Invalid

L8, P7: 0:59.688
S1(24.24)  S2(20.10)  S3(15.34)

L9, P8: Invalid

